> **Пересчёт.** Основа — `petri_states_flags.ipynb` из папки `петри`; код расчёта тот же, пути и флаги — из `common/settings.py`. Перебор разделён по частям: в каждой только свои пары флагов; шаги 1, 3, 4, 6 считаются по всей базе и в обеих частях дают одно и то же. Текст с числами и выводами написан по **прежнему** расчёту: после пересчёта сверить (`common/compare_with_old.py`).

# Журнал состояний: подбор флагов `SHORT_CUSTODY_DAYS` и `LONG_CUSTODY_DAYS`

К `petri_states.ipynb`, `petri_states_eval_2013.ipynb`, `petri_states_eval_all.ipynb` (04.11.2026).

| Флаг | Было | Правило |
|---|---|---|
| `SHORT_CUSTODY_DAYS` | 15 | 6: заключение короче — аресты/обвинения внутри срока склеиваются с ним (`jail` → `charge+jail`); иначе правило 7: `jail → дело → jail` |
| `LONG_CUSTODY_DAYS` | 365 | 5: заключение не короче — аресты/обвинения дня входа отдельным состоянием перед ним; иначе правило 3: составное `arrest+jail` |

Шаги:
1. медианы длительности заключения по зонам правил 3–7;
2. медианы времени на переходах журнала состояний (окно 2013, текущие флаги);
3. граница `LONG` по данным — где эпизод из «задержания по делу дня входа» становится «сроком»;
4. граница `SHORT` по данным — до какой длительности события внутри срока — продолжение оформления задержания;
5. перебор флагов по качеству сетей (train/test 80/20, TBR fitness, ETC precision; без alignments).

## Шаг 0. Настройки

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "all"                         # "2013" — с окном, "all" — без окна

DB_CANDIDATES = cfg.DB_CANDIDATES
CATEGORIES_DIRS = cfg.CATEGORIES_DIRS

SHORT_CUSTODY_DAYS = cfg.CONFIGS["long365_v1"]["short"]     # значения флагов до подбора
LONG_CUSTODY_DAYS = cfg.CONFIGS["long365_v1"]["long"]
LOG_START = cfg.PARTS[PART]["log_start"]         # окно для шага 2 и перебора

RUN_SWEEP = cfg.RUN_V1           # перебор считает метрики по варианту 1: выключен вместе с ним (RUN_V1 в settings.py)
OUT = cfg.out_dir(PART, "flags")
SWEEP_FILE = OUT / "flags_sweep.jsonl"


In [2]:
import sys, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
import petri_pipeline as pp
import petri_states as ps
import flags_analysis as fa

pd.set_option("display.width", 250)
OUT.mkdir(parents=True, exist_ok=True)
DB = pp.find_db(DB_CANDIDATES)
categorize = pp.import_categorizer(CATEGORIES_DIRS)
tables, _ = pp.drop_full_duplicates(pp.load_tables(DB))
tables, _, _ = pp.drop_invalid_stays(tables)
hist = pp.build_history_table(tables, categorize)
obs_end = pp.observation_end(hist)
eps, inside = fa.episode_profile(hist, obs_end)
print(f"эпизодов заключения: {len(eps):,}; единиц дел внутри срока: {len(inside):,} в {inside.episode_id.nunique():,} эпизодах")

эпизодов заключения: 25,231; единиц дел внутри срока: 1,790 в 1,295 эпизодах


## Шаг 1. Медианы длительности заключения по зонам правил

Эпизод — склеенные пребывания jail/prison (как в `petri_states.build_state_log`); кто сидит на конец наблюдения — до конца наблюдения. Дни.

In [3]:
t1 = fa.domain_medians(eps, SHORT_CUSTODY_DAYS, LONG_CUSTODY_DAYS)
t1.to_csv(OUT / "domain_medians_current.csv")
display(t1)

,n,q25,median,q75,доля prison
"зона (SHORT=15, LONG=365)",,,,,
все эпизоды,25231,1.06,7.31,81.96,0.19
"3: одно дело в день входа, < LONG (составное)",13230,0.97,2.02,19.36,0.01
4: несколько дел в день входа,4387,1.20,12.60,64.95,0.08
"5: одно дело в день входа, ≥ LONG (дело → заключение)",327,470.01,609.56,827.99,0.87
без дела в день входа,7287,4.83,117.76,433.00,0.57
"6: события внутри срока, < SHORT (склейка)",154,2.94,6.04,9.52,0.01
"7: события внутри срока, ≥ SHORT (jail → дело → jail)",1141,54.01,156.62,479.00,0.34


## Шаг 2. Медианы на переходах журнала состояний (вся история, текущие флаги)

Время от состояния до следующего (`dt_next_days`), дни.

In [4]:
log, _, rep = ps.build_state_log(hist, obs_end=obs_end, log_start=LOG_START,
                                 short_days=SHORT_CUSTODY_DAYS, long_days=LONG_CUSTODY_DAYS)
t2 = fa.transition_medians(log)
t2.to_csv(OUT / "transition_medians_current.csv")
display(t2)

,n,q25,median,q75
переход,,,,
"составное дело+заключение → следующее (правила 3, 6)",13296,0.95,1.96,15.20
"дело → jail/prison в тот же день (правила 4, 5)",4688,0.17,0.29,0.42
jail/prison без дела → следующее,11935,1.98,36.01,229.28
заключение → дело внутри срока (правило 7),1615,1.82,17.77,61.92
дело внутри срока → возврат (правило 7),1609,1.00,1.00,1.00
возврат → следующее (правило 7),1374,20.32,60.50,174.88


## Шаг 3. Граница `LONG_CUSTODY_DAYS` по данным

По длительности эпизода: доля эпизодов с prison и доля с делом в день входа. Короткие эпизоды — задержание по делу дня входа (jail), длинные — срок (prison), дела в день входа обычно нет. Правило 5 нужно там, где начинается «срок».

In [5]:
t3 = fa.long_profile(eps)
t3.to_csv(OUT / "long_profile.csv")
display(t3)
print("prison > 50% — с", fa.first_crossing(t3, "доля_prison", above=True), "дней;",
      "дело в день входа < 50% — с", fa.first_crossing(t3, "доля_с_делом_в_день_входа", above=False), "дней")

,эпизодов,доля_prison,доля_с_делом_в_день_входа,доля_одно_дело
dur,,,,
"[0.0, 30.0)",15767,0.010,0.836,0.669
"[30.0, 60.0)",2294,0.058,0.797,0.537
"[60.0, 90.0)",1088,0.188,0.698,0.437
"[90.0, 120.0)",747,0.299,0.600,0.380
"[120.0, 150.0)",476,0.443,0.515,0.298
"[150.0, 180.0)",405,0.405,0.533,0.309
"[180.0, 210.0)",350,0.520,0.466,0.291
"[210.0, 240.0)",336,0.625,0.411,0.247
"[240.0, 270.0)",358,0.564,0.453,0.274


prison > 50% — с 180.0 дней; дело в день входа < 50% — с 180.0 дней


## Шаг 4. Граница `SHORT_CUSTODY_DAYS` по данным

События внутри срока по длительности заключения: доля на следующий день после входа (продолжение оформления задержания — склейка уместна) и медиана дней от входа.

In [6]:
t4 = fa.short_profile(inside)
t4.to_csv(OUT / "short_profile.csv")
display(t4)
b = t4[t4["доля_на_след_день"] < 0.5]
print("на следующий день < 50% — с", b.index[0].left if len(b) else None, "дней")

,единиц,доля_на_след_день,медиана_дней_от_входа,доля_то_же_дело
dur,,,,
"[0.0, 3.0)",51,0.941,1.0,0.118
"[3.0, 7.0)",59,0.644,1.0,0.119
"[7.0, 10.0)",30,0.733,1.0,0.067
"[10.0, 15.0)",41,0.561,1.0,0.244
"[15.0, 20.0)",37,0.432,3.0,0.189
"[20.0, 30.0)",66,0.424,3.0,0.136
"[30.0, 45.0)",168,0.327,9.0,0.167
"[45.0, 60.0)",97,0.258,12.0,0.206
"[60.0, 90.0)",145,0.290,11.0,0.166


на следующий день < 50% — с 15.0 дней


## Шаг 5. Перебор флагов по качеству сетей

Окно 2013, люди train/test 80/20 (seed 42; часть пар — ещё seed 7 и вся история). Сеть на train, на test — TBR fitness и ETC precision; `F1 = 2·f·p/(f+p)`, `Pnorm` — precision, нормированная на цветочную сеть. Alignments не считаются. Модели: DFG p0.05, HM 0.95, Split Miner.

In [7]:
if RUN_SWEEP:
    for seed, pairs in cfg.PARTS[PART]["sweep"].items():
        fa.sweep(hist, pairs, SWEEP_FILE, log_start=LOG_START, seed=seed)
    t5 = fa.sweep_table(SWEEP_FILE)
    t5.to_csv(OUT / "flags_sweep_table.csv")
    print("вся история, seed 42")
    display(t5)
else:
    print("перебор флагов выключен: RUN_V1 = False в common/settings.py")

replaying log with TBR, completed traces ::   0%|          | 0/1631 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/30598 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/1631 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/30598 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/1631 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/30598 [00:00<?, ?it/s]

... (19 outputs left out)

replaying log with TBR, completed traces ::   0%|          | 0/30452 [00:00<?, ?it/s]

SHORT=15, LONG=365: 243 с


вся история, seed 42


F1 DFG_p0.05  F1 HM_dep0.95  F1 SplitMiner  F1 среднее  Pnorm DFG_p0.05  Pnorm HM_dep0.95  Pnorm SplitMiner  variants_test  composite  absorbed  inside
exp seed short long                                                                                                                                                         
all 42   5     180         0.7528         0.5254         0.7851      0.6878           0.5220            0.1937            0.7619           1631      12806        76    1714
         15    180         0.7527         0.5252         0.7985      0.6921           0.5220            0.1934            0.8454           1630      12806       181    1609
               365         0.7524         0.5254         0.7934      0.6904           0.5312            0.2093            0.8566           1626      13228       181    1609

## Шаг 6. Медианы при рекомендованных флагах

In [8]:
NEW_SHORT, NEW_LONG = 15, 180
t6 = fa.domain_medians(eps, NEW_SHORT, NEW_LONG)
t6.to_csv(OUT / "domain_medians_recommended.csv")
display(t6)

,n,q25,median,q75,доля prison
"зона (SHORT=15, LONG=180)",,,,,
все эпизоды,25231,1.06,7.31,81.96,0.19
"3: одно дело в день входа, < LONG (составное)",12808,0.95,1.93,15.16,0.00
4: несколько дел в день входа,4387,1.20,12.60,64.95,0.08
"5: одно дело в день входа, ≥ LONG (дело → заключение)",749,240.12,330.93,564.48,0.53
без дела в день входа,7287,4.83,117.76,433.00,0.57
"6: события внутри срока, < SHORT (склейка)",154,2.94,6.04,9.52,0.01
"7: события внутри срока, ≥ SHORT (jail → дело → jail)",1141,54.01,156.62,479.00,0.34


## Выводы

**`LONG_CUSTODY_DAYS`: 365 → 180.**
- По данным граница между «задержанием по делу дня входа» и «сроком» — 180 дней: с 180 дней доля эпизодов с prison больше 50%, а доля эпизодов с делом в день входа меньше 50% (шаг 3).
- При 365 правило 5 срабатывает в 327 эпизодах, медиана их длительности — 610 дней (87% с prison). При 180 — в 749 эпизодах, медиана 331 день (53% с prison). Остальные эпизоды с одним делом в день входа (составное состояние, правило 3): медиана 2 дня. На переходах (шаг 2): дело → заключение в тот же день — медиана 0.3 дня (7 ч от даты дела до входа), составное состояние → следующее — 2 дня.
- Метрики (шаг 5): от 180 до 365 дней — плато (средний F1 0.791–0.793), меньше 90 дней — F1 DFG и HM ниже, 545 дней и больше — F1 Split Miner падает с 0.78 до 0.72. По всей истории 180 немного лучше 365 (средний F1 0.692 против 0.690), с seed 7 — на уровне шума (0.7906 против 0.7916). Получается, метрики допускают весь диапазон 180–365, а выбор внутри него делается по данным.

**`SHORT_CUSTODY_DAYS`: оставить 15.**
- События внутри срока при заключении до 15 дней — в основном на следующий день после входа (56–94%) с медианой 1 день от входа: это продолжение оформления задержания, склейка уместна. С 15 дней доля на следующий день меньше 50%, медиана — 3 дня и больше (шаг 4).
- В зоне правила 6 (короче 15 дней) — 154 эпизода, медиана 6 дней; в зоне правила 7 — 1 141 эпизод, медиана 157 дней. На переходах: дело внутри срока → возврат в заключение — медиана 1 день, возврат → следующее — 53 дня (шаг 2).
- Метрики к `SHORT` почти не чувствительны: от 0 до 60 дней средний F1 меняется меньше чем на 0.005, и лучшая точка разная у разных seed (затронуто до 549 единиц из ~117 тыс. состояний). То есть метрики выбор не определяют, решает профиль данных.